# QTAP: Quantum Torsion Angle Predictor
Variational quantum Born machine for Ramachandran phi/psi distribution prediction.

**Author:** Tommaso R. Marena, Catholic University of America, 2026

**Pipeline:**
1. Install dependencies
2. Imports and configuration
3. Ramachandran reference distributions
4. Quantum Born machine circuit
5. Residue feature encoder
6. Forward pass + KL loss
7. COBYLA training loop
8. Classical MLP baseline
9. Visualization and results

In [ ]:
import subprocess, sys
pkgs = ['qiskit>=1.0.0', 'qiskit-aer>=0.14.0', 'scipy', 'matplotlib', 'pandas', 'torch', 'tqdm']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
print('All dependencies installed.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.optimize import minimize
from scipy.special import rel_entr
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
NQ = 4
NB = 16
DEPTH = 3
SHOTS = 1024
STEPS = 180
sim = AerSimulator()
AA = list('ACDEFGHIKLMNPQRSTVWY')
print(f'Config: {NQ} qubits, {NB} bins, depth={DEPTH}, shots={SHOTS}')

In [ ]:
def refdist(a):
    d = np.zeros(16)
    if a == 'G':
        d[:] = 1 / 16
    elif a == 'P':
        d[[5, 13, 4]] = [0.60, 0.30, 0.10]
    elif a in 'AVILMFYW':
        d[[5, 12, 13, 4]] = [0.40, 0.35, 0.15, 0.10]
    elif a in 'DENQ':
        d[[5, 12, 13, 1]] = [0.50, 0.25, 0.15, 0.10]
    elif a in 'KRH':
        d[[5, 12, 13, 10]] = [0.35, 0.30, 0.20, 0.15]
    elif a in 'ST':
        d[[12, 5, 13]] = [0.45, 0.30, 0.25]
    else:
        d[[5, 12, 13, 4]] = [0.38, 0.32, 0.18, 0.12]
    d += 1e-6
    return d / d.sum()

refs = {a: refdist(a) for a in AA}
print('Reference distributions built for', len(refs), 'residues.')

In [ ]:
AA_PROPS = {
    'A': (89.09,  1.8,  6.00,  0.0, 0.0),
    'C': (121.16, 2.5,  5.07,  0.0, 0.0),
    'D': (133.10,-3.5,  2.77, -1.0, 0.0),
    'E': (147.13,-3.5,  3.22, -1.0, 0.0),
    'F': (165.19, 2.8,  5.48,  0.0, 1.0),
    'G': (75.03, -0.4,  5.97,  0.0, 0.0),
    'H': (155.16,-3.2,  7.59,  0.1, 1.0),
    'I': (131.17, 4.5,  6.02,  0.0, 0.0),
    'K': (146.19,-3.9, 10.53,  1.0, 0.0),
    'L': (131.17, 3.8,  5.98,  0.0, 0.0),
    'M': (149.21, 1.9,  5.74,  0.0, 0.0),
    'N': (132.12,-3.5,  5.41,  0.0, 0.0),
    'P': (115.13,-1.6,  6.30,  0.0, 0.0),
    'Q': (146.15,-3.5,  5.65,  0.0, 0.0),
    'R': (174.20,-4.5, 10.76,  1.0, 0.0),
    'S': (105.09,-0.8,  5.68,  0.0, 0.0),
    'T': (119.12,-0.7,  5.60,  0.0, 0.0),
    'V': (117.15, 4.2,  5.96,  0.0, 0.0),
    'W': (204.23,-0.9,  5.89,  0.0, 1.0),
    'Y': (181.19,-1.3,  5.66,  0.0, 1.0),
}
RANGES = [(75.03,204.23),(-4.5,4.5),(2.77,10.76),(-1,1),(0,1)]

def encode(a):
    return np.array([(AA_PROPS[a][i]-RANGES[i][0])/(RANGES[i][1]-RANGES[i][0])*2*np.pi
                     for i in range(NQ)])

enc = {a: encode(a) for a in AA}
print('Encoding angles ready.')

In [ ]:
def build_circuit(nq=NQ, depth=DEPTH):
    e = ParameterVector('enc', nq)
    v = ParameterVector('var', 2*nq*depth)
    qc = QuantumCircuit(nq)
    for i in range(nq):
        qc.ry(e[i], i)
    k = 0
    for _ in range(depth):
        for i in range(nq):
            qc.ry(v[k], i); k += 1
        for i in range(nq):
            qc.rz(v[k], i); k += 1
        for i in range(nq - 1):
            qc.cx(i, i+1)
    qc.measure_all()
    return qc, e, v

qc, epar, vpar = build_circuit()
print(qc.draw(output='text'))
print('Variational parameters:', len(vpar))

In [ ]:
def born(x, theta, shots=SHOTS):
    bind = {p: float(x[i]) for i, p in enumerate(epar)}
    bind.update({p: float(theta[i]) for i, p in enumerate(vpar)})
    counts = sim.run(transpile(qc.assign_parameters(bind), sim), shots=shots).result().get_counts()
    p = np.zeros(2**NQ)
    for b, c in counts.items():
        p[int(b, 2)] += c
    p = p / p.sum() + 1e-9
    return p / p.sum()

def KL(a, b):
    return float(np.sum(rel_entr(a, b)))

theta_test = np.random.uniform(0, 2*np.pi, len(vpar))
p_test = born(enc['A'], theta_test)
print('Forward pass OK. KL(Ala):', round(KL(refs['A'], p_test), 4))

In [ ]:
def train(a, steps=STEPS):
    th0 = np.random.uniform(0, 2*np.pi, len(vpar))
    hist = []
    def obj(th):
        loss = KL(refs[a], born(enc[a], th))
        hist.append(loss)
        return loss
    r = minimize(obj, th0, method='COBYLA', options={'maxiter': steps, 'rhobeg': 0.5})
    return {'kl': float(r.fun), 'theta': r.x, 'hist': hist}

targets = ['A', 'G', 'P', 'V']
res = {a: train(a) for a in tqdm(targets, desc='Training QTAP')}
print({a: round(res[a]['kl'], 3) for a in targets})

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 16), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.net(x)

mlp = MLP()
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
lossfn = nn.KLDivLoss(reduction='batchmean')

X = torch.tensor([[AA_PROPS[a][i] / max(abs(RANGES[i][0]), abs(RANGES[i][1]))
                    for i in range(5)] for a in targets], dtype=torch.float32)
Y = torch.tensor([refs[a] for a in targets], dtype=torch.float32)

for _ in range(1500):
    opt.zero_grad()
    pred = mlp(X)
    loss = lossfn(torch.log(pred + 1e-9), Y)
    loss.backward()
    opt.step()

with torch.no_grad():
    mp = mlp(X).numpy()

rows = [[a, res[a]['kl'], KL(refs[a], mp[i]),
         'QTAP' if res[a]['kl'] <= KL(refs[a], mp[i]) else 'MLP']
        for i, a in enumerate(targets)]
df = pd.DataFrame(rows, columns=['Residue', 'QTAP_KL', 'MLP_KL', 'Winner'])
df

In [ ]:
import os, json
os.makedirs('qtap_outputs', exist_ok=True)
labels = ['-180', '-90', '0', '+90']

for i, a in enumerate(targets):
    qp = born(enc[a], res[a]['theta'], shots=4096)
    grids = [refs[a].reshape(4, 4), qp.reshape(4, 4), mp[i].reshape(4, 4)]
    fig, axs = plt.subplots(1, 3, figsize=(12, 3.5))
    fig.suptitle(f'Ramachandran: {a}')
    for ax, g, t in zip(axs, grids, ['Reference', 'QTAP', 'MLP']):
        ax.imshow(g, origin='lower', cmap='hot_r',
                  vmin=0, vmax=max(x.max() for x in grids))
        ax.set_title(t)
        ax.set_xticks(range(4)); ax.set_xticklabels(labels)
        ax.set_yticks(range(4)); ax.set_yticklabels(labels)
        for r in range(4):
            for c in range(4):
                ax.text(c, r, f'{g[r,c]:.2f}', ha='center', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(f'qtap_outputs/ramachandran_{a}.png', dpi=150)
    plt.show()

df.to_csv('qtap_outputs/results.csv', index=False)
with open('qtap_outputs/results.json', 'w') as f:
    json.dump({a: {'qtap_kl': res[a]['kl']} for a in targets}, f, indent=2)
print('Saved to qtap_outputs/')